# Introduction to Calendar Spreads

A calendar spread is an options trading strategy that involves buying and selling options with the same strike price but different expiration dates. This strategy takes advantage of the differences in time decay (theta) between short-term and long-term options.

## What Is a Calendar Spread?

In a typical calendar spread, a trader will:

- Sell a short-term option – This option has a nearer expiration date and decays faster as it approaches expiration.
- Buy a long-term option – This option has a later expiration date and retains its value longer due to slower time decay.

By structuring the trade this way, traders can potentially profit from the faster time decay of the short-term option while still maintaining a long-term position at a lower cost.

## Understanding Key Terms

To have a good understanding of the strategy, it’s important to understand a few key concepts:

- Strike (Exercise) Price – The predetermined price at which the option contract can be exercised.
- Expiration Date (Days to Expiration) – The date on which the option contract expires. Short-term options expire sooner than long-term options.
- Short-Term Option – An option contract with an expiration date that is closer to the present, such as a few days, weeks, or months.
- Long-Term Option – An option contract with an expiration date further in the future, sometimes several months or years away.
- Time Decay (Theta) – A measure of how an option’s price decreases over time. Short-term options lose value more quickly than long-term options.

## When to Use a Calendar Spread

The strategy is especially useful when:

1. You expect the price to stay near the strike around the short-term option’s expiration.
2. You want to profit from the faster time decay of the short-term option.
3. You exploit theta decay: the short option loses value quickly, while the long option decays more slowly.


## Profit and Risk Profile

### You Profit When:
- The underlying price stays close to the strike at the short option’s expiration.
- You gain from the premium decay of the short leg, while the long leg retains value.

### Risk:
- Limited loss: the maximum you can lose is the net premium paid for the spread.
- Losses occur if the price moves far from the strike, making both options lose value.


## Calendar Spread and Margins

Beyond its strategic flexibility, the calendar spread also has important implications for margin efficiency. Because the long and short positions partially offset each other’s risks, the overall margin required to hold a calendar spread is often significantly lower than the sum of its parts.

- Selling a naked option carries theoretical unlimited risk - brokers require large margin reserves.
- However, pairing that short option with a longer-dated long option at the same strike limits the risk.
- Brokers recognize this as a hedged position and significantly reduce the required margin.

This makes the calendar spread not just a trading strategy, but also a capital optimization tool.

Cloud Prisma Margin Estimator is especially valuable for exploring margin impacts of calendar spreads, enabling traders to:

- Quantify the margin savings from expiry mismatch and risk offsetting,
- Compare different calendar spread configurations (e.g. 1-month vs. 2-month vs. 3-month),
- Evaluate how volatility, expiry dates, and strikes affect capital requirements.

In this tutorial, we'll demonstrate how to construct and analyze a calendar spread using CPME, helping traders understand not just the trading mechanics, but also the margin profile of the strategy.

## Data Retrieval

After describing the Calendar Spread Strategy, let's now see how we can leverage our tool to implement it. The first step is the retrieval of the data with our requests. We will implement the strategy on the “OESX” product.
First, let's fetch the series for OESX. We will make two requests, one for short-term options with expiration date less than 30 days, and another one for long-term options with expiration date of more than 30 days. For this filtering, we will use option `max_tte` and `min_tte`, which filter the returned series by expiration date.
Calendar spread needs the exerces price (strike price) to be the same, so we will filter it by the same value. 
We will also need to generate the template for the processing for our estimator endpoint, so that the margins can be calculated.

In [3]:
!margin_estimator_tool get_series \
    --version LIVE \
    --products OESX \
    --max_tte 30 \
    --export_dir /home/gk101/workspace/DAVe-MarginEstimator-PythonAPIClient/margin_estimator_tool/output \
    --filter exercise_price:4200 \
    --template

2025-04-09 18:52:39,560 api-client INFO   cpme_api configuration.enable_logging   Logger start.
2025-04-09 18:52:39,565 api-client INFO   cpme_api rest.GET              /series	HEADER:{'Content-Type': 'application/json', 'X-DBP-APIKEY': '9c40a29c-?..', 'User-Agent': 'cpME-api/3.0.0/python'} PARAMS:[('products', 'OESX'), ('extrafields', 'product_id,contract_date,contract_maturity,expiry_maturity,call_put_flag,exercise_price,version_number,iid,act_trade_unit_no,days_to_expiration,trade_unit_value,exercise_style_flag,contract_frequency'), ('business_date', 20250409), ('live_timestamp', 0), ('live', True)] URL:https://risk.developer.deutsche-boerse.com/prisma-margin-estimator-2-0-0/series?products=OESX&extrafields=product_id%2Ccontract_date%2Ccontract_maturity%2Cexpiry_maturity%2Ccall_put_flag%2Cexercise_price%2Cversion_number%2Ciid%2Cact_trade_unit_no%2Cdays_to_expiration%2Ctrade_unit_value%2Cexercise_style_flag%2Ccontract_frequency&business_date=20250409&live_timestamp=0&live=True 
Reque

In [2]:
!margin_estimator_tool get_series \
    --version LIVE \
    --products OESX \
    --min_tte 30 \
    --export_dir /home/gk101/workspace/DAVe-MarginEstimator-PythonAPIClient/margin_estimator_tool \
    --filter exercise_price:4200 \
    --template

2025-04-09 18:52:31,086 api-client INFO   cpme_api configuration.enable_logging   Logger start.
2025-04-09 18:52:31,091 api-client INFO   cpme_api rest.GET              /series	HEADER:{'Content-Type': 'application/json', 'X-DBP-APIKEY': '9c40a29c-?..', 'User-Agent': 'cpME-api/3.0.0/python'} PARAMS:[('products', 'OESX'), ('extrafields', 'product_id,contract_date,contract_maturity,expiry_maturity,call_put_flag,exercise_price,version_number,iid,act_trade_unit_no,days_to_expiration,trade_unit_value,exercise_style_flag,contract_frequency'), ('business_date', 20250409), ('live_timestamp', 0), ('live', True)] URL:https://risk.developer.deutsche-boerse.com/prisma-margin-estimator-2-0-0/series?products=OESX&extrafields=product_id%2Ccontract_date%2Ccontract_maturity%2Cexpiry_maturity%2Ccall_put_flag%2Cexercise_price%2Cversion_number%2Ciid%2Cact_trade_unit_no%2Cdays_to_expiration%2Ctrade_unit_value%2Cexercise_style_flag%2Ccontract_frequency&business_date=20250409&live_timestamp=0&live=True 
Reque

Now we have our options filtered down. Let's see how the templates look like:

In [4]:
with open("../output/20250409_LIVE_etd_portfolio_template.csv", "r") as f:
    print(f.read())

Product ID,Contract Date,Call Put Flag,Exercise Price,Version Number,Net LS Balance
OESX,20250417,C,4200.0,0,
OESX,20250417,P,4200.0,0,
OESX,20250411,P,4200.0,0,
OESX,20250425,C,4200.0,0,
OESX,20250425,P,4200.0,0,
OESX,20250411,C,4200.0,0,
OESX,20250502,C,4200.0,0,
OESX,20250430,C,4200.0,0,
OESX,20250502,P,4200.0,0,
OESX,20250509,C,4200.0,0,
OESX,20250509,P,4200.0,0,
OESX,20250430,P,4200.0,0,



In [5]:
with open("../20250409_LIVE_etd_portfolio_template.csv", "r") as f:
    print(f.read())

Product ID,Contract Date,Call Put Flag,Exercise Price,Version Number,Net LS Balance
OESX,20261218,C,4200.0,0,
OESX,20251219,P,4200.0,0,
OESX,20261218,P,4200.0,0,
OESX,20251219,C,4200.0,0,
OESX,20271217,P,4200.0,0,
OESX,20271217,C,4200.0,0,
OESX,20281215,P,4200.0,0,
OESX,20281215,C,4200.0,0,
OESX,20291221,C,4200.0,0,
OESX,20291221,P,4200.0,0,
OESX,20301220,P,4200.0,0,
OESX,20301220,C,4200.0,0,
OESX,20311219,P,4200.0,0,
OESX,20311219,C,4200.0,0,
OESX,20250620,P,4200.0,0,
OESX,20250620,C,4200.0,0,
OESX,20250919,P,4200.0,0,
OESX,20250919,C,4200.0,0,
OESX,20321217,P,4200.0,0,
OESX,20321217,C,4200.0,0,
OESX,20260320,P,4200.0,0,
OESX,20260320,C,4200.0,0,
OESX,20260619,C,4200.0,0,
OESX,20260619,P,4200.0,0,
OESX,20260918,C,4200.0,0,
OESX,20260918,P,4200.0,0,
OESX,20331216,P,4200.0,0,
OESX,20331216,C,4200.0,0,
OESX,20270319,C,4200.0,0,
OESX,20270319,P,4200.0,0,
OESX,20250516,C,4200.0,0,
OESX,20250516,P,4200.0,0,
OESX,20270618,C,4200.0,0,
OESX,20270618,P,4200.0,0,
OESX,20250718,P,4200.0,0,
OESX,2

For our purposes, picking only one option from each portfolio is sufficient. Let's create our portfolio now. We will take one short option that we will sell and one long option that we would buy. This is how our portfolio will look like:

In [6]:
with open("../output/20250409_LIVE_etd_portfolio_template.csv", "r") as f:
    print(f.read())

Product ID,Contract Date,Call Put Flag,Exercise Price,Version Number,Net LS Balance
OESX,20250417,C,4200.0,0,-1
OESX,20261218,C,4200.0,0,1



Now, let's send it to estimator endpoint to process the margins.

In [2]:
!margin_estimator_tool etd_portfolio \
    --version LIVE \
    --csv_file ../output/20250409_LIVE_etd_portfolio_template.csv \
    --to_json \
    --export_dir /home/gk101/workspace/DAVe-MarginEstimator-PythonAPIClient/margin_estimator_tool/output

2025-04-09 23:08:39,473 api-client INFO   cpme_api configuration.enable_logging   Logger start.
Headers validated successfully.
2025-04-09 23:08:39,478 api-client INFO   cpme_api rest.POST             /estimator	HEADER:{'Content-Type': 'application/json', 'X-DBP-APIKEY': '9c40a29c-?..', 'User-Agent': 'cpME-api/3.0.0/python'} PARAMS:[] URL:https://risk.developer.deutsche-boerse.com/prisma-margin-estimator-2-0-0/estimator 
Request successful.
Portfolio exported to /home/gk101/workspace/DAVe-MarginEstimator-PythonAPIClient/margin_estimator_tool/output


## Output Interpretation

Let's now analyze the output. We will frist load the portfolio.

In [4]:
import json
import pandas as pd

with open('../output/20250409_LIVE_estimator.json', 'r') as file:
    data = json.load(file)

# Extract portfolio margin details
portfolio_margin = data["portfolio_margin"][0]

initial_margin = portfolio_margin['initial_margin']
market_risk = portfolio_margin['market_risk']
premium_margin = portfolio_margin['premium_margin']

# Display the portfolio margin summary
portfolio_margin_summary = {
    "Initial Margin": initial_margin,
    "Market Risk": market_risk,
    "Premium Margin": premium_margin,
}

portfolio_margin_summary_df = pd.DataFrame(portfolio_margin_summary, index=[0])
portfolio_margin_summary_df

,Initial Margin,Market Risk,Premium Margin
0,895.414851,895.41118,-1953.3


In [5]:
# Extract and display the drilldown data for each instrument
drilldown_data = data["drilldowns"]
drilldown_list = []

for d in drilldown_data:
    drilldown_list.append({
        "Product ID": d["product_id"],
        "Contract Date": d["contract_date"],
        "Type": "Call" if d["call_put_flag"] == "C" else "Put",
        "Exercise Price": d["exercise_price"],
        "Position": "Long" if d["net_ls_balance"] > 0 else "Short",
        "Component Margin": d["component_margin"],
        "Premium Margin": d["premium_margin"]
    })

drilldown_df = pd.DataFrame(drilldown_list)
drilldown_df

,Product ID,Contract Date,Type,Exercise Price,Position,Component Margin,Premium Margin
0,OESX,20261218,Call,4200.0,Long,-2179.235303,-7755.8
1,OESX,20250417,Call,4200.0,Short,3074.650154,5802.5


In a calendar spread, the margin analysis highlights the effects of the short-term sold option and the long-term bought option. The negative margin for the long-term option is offsetting the margin requirement for the short-term option, reducing the overall margin for the entire calendar spread. This structure of the portfolio helps manage risk and capital efficiency, with the premium margin indicating the income from selling the short-term option. The net margin reflects the hedged nature of the position, with an overall reduced margin requirement compared to outright options trades.